# D1-12 Optional - Correlated water balances in Monte Carlo

This optional notebook shows how to preserve process-level water balances during Brightway Monte Carlo calculations.

Standard `bw2calc.LCA(..., use_distributions=True)` samples each uncertain matrix element independently. That is usually fine, but it can break a process-level balance when water withdrawals and water releases should move together.

Here we scan the active product system, pre-sample water withdrawals for every process with both withdrawals and releases, derive the matching releases from the sampled total withdrawal, and inject all these sample series into `bw2calc.LCA` with a `bw_processing` datapackage.


## Learning goals

After this notebook, you should be able to:

- Find every process in a product system with water withdrawals and releases.
- Show why independent Monte Carlo sampling can break process-level return fractions.
- Build a sequential `bw_processing` datapackage with system-wide correlated water samples.
- Keep normal Brightway uncertainty sampling for all other exchanges.


## 1) Imports and project setup

We use the `aalborg-rlcia-2026` Brightway project and the bundled `bafu` database.


In [ ]:
import warnings

# Keep notebook output focused. Stochastic Brightway solves can emit repeated
# UMFPACK conditioning warnings; a long warning stream can make JupyterLab
# report "Page unresponsive" even when the calculation itself is progressing.
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API.*")
try:
    from scikits.umfpack.umfpack import UmfpackWarning
    warnings.filterwarnings("ignore", category=UmfpackWarning)
except Exception:
    UmfpackWarning = None

# `defaultdict` is used to aggregate repeated exchanges that map to the
# same matrix coordinate: same biosphere flow, same process column.
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from stats_arrays import UncertaintyBase, uncertainty_choices

import bw2calc as bc
import bw2data as bd
import bw_processing as bwp

pd.options.display.float_format = "{:,.6g}".format

bd.projects.set_current("aalborg-rlcia-2026")
database_name = "bafu"
method = ('EN15804', 'inventory indicators ISO21930', 'use of net fresh water')

# Keep the example quick. Increase this only when you really need more draws.
iterations = 300
seed = 1234

# Treat very small supply-array entries as numerical zero.
active_supply_cutoff = 1e-20

In [ ]:
# Quick check: this inventory indicator gives +1 to water withdrawals and -1
# to water releases. This is why inconsistent Monte Carlo water balances can
# directly affect the score distribution.
for flow_id, cf in bd.Method(method).load():
    flow = bd.get_activity(flow_id)
    print(flow["name"], flow["categories"], cf)

## 2) Choose a functional unit and build the product system

The functional unit is still one BAFU activity. The important change is that the water-balance correction is not limited to this activity. After building the LCA once, we inspect all activity columns with non-zero supply in this product system.


In [ ]:
db = bd.Database("bafu")

# Pick any BAFU activity as the functional unit. The correction below is not
# restricted to this foreground process; it is applied to all active processes
# in the solved product system that have both water withdrawals and releases.
activity = [
    act for act in db
    if all(
        x in act["name"].lower()
        for x in ("photovoltaic", "panel")
    )
    and act["unit"] == "square meter"
][0]
print(activity)

In [ ]:
# `prepare_lca_inputs` gives us the integer-mapped demand, datapackages, and
# remapping dictionaries expected by `bw2calc.LCA`.
demand, data_objs, remapping = bd.prepare_lca_inputs(demand={activity: 1}, method=method)

# The deterministic LCA is used for two things:
# 1. building the product system and supply array;
# 2. reading the deterministic biosphere matrix for candidate filtering.
base_lca = bc.LCA(demand, data_objs=data_objs, remapping_dicts=remapping)
base_lca.lci()
base_lca.lcia()

active_activity_ids = [
    activity_id
    for activity_id, column in base_lca.dicts.activity.items()
    if abs(float(base_lca.supply_array[column])) > active_supply_cutoff
]

print("Functional unit:", activity)
print("Activity columns in matrix:", len(base_lca.dicts.activity))
print("Active activity columns in this product system:", len(active_activity_ids))
print("Deterministic score:", base_lca.score)


## 3) Find candidate processes from `biosphere_matrix`

The fastest first filter is the sparse `biosphere_matrix`. We first classify biosphere rows as water withdrawals or releases, then find activity columns with non-zero entries in both sets of rows.

The flow selectors are intentionally broad for this teaching example:

- withdrawal: cubic-meter biosphere flow whose first category is `natural resource` and whose name contains `water`
- release: cubic-meter biosphere flow named `Water` whose first category is `water` or `air`

After this matrix filter, we inspect only the candidate activities with `.biosphere()` because the uncertainty parameters live on exchanges, not in the matrix itself. For each candidate process, the deterministic return fraction is:

`sum(release amounts) / sum(withdrawal amounts)`


In [ ]:
# These selectors work on biosphere flow nodes, not on exchanges.
# They define the water accounting convention for this notebook.
def is_water_withdrawal(flow):
    categories = tuple(flow.get("categories") or ())
    name = (flow.get("name") or "").lower()
    return (
        flow.get("unit") == "cubic meter"
        and categories[:1] == ("natural resource",)
        and "water" in name
    )


def is_water_release(flow):
    categories = tuple(flow.get("categories") or ())
    return (
        flow.get("unit") == "cubic meter"
        and flow.get("name") == "Water"
        and bool(categories)
        and categories[0] in {"water", "air"}
    )


# Brightway matrices are integer-indexed. The dictionaries map between
# Brightway node ids and matrix row/column positions.
def flow_ids_matching(selector):
    return [
        flow_id
        for flow_id in base_lca.dicts.biosphere
        if selector(bd.get_node(id=flow_id))
    ]


def matrix_rows(flow_ids):
    return [base_lca.dicts.biosphere[flow_id] for flow_id in flow_ids]


def nonzero_columns(rows):
    # `matrix[rows, :].nonzero()[1]` returns the activity columns where any
    # selected biosphere row has a non-zero exchange amount.
    if not rows:
        return set()
    return set(base_lca.biosphere_matrix[rows, :].nonzero()[1])


# First classify the biosphere matrix rows once.
water_withdrawal_flow_ids = flow_ids_matching(is_water_withdrawal)
water_release_flow_ids = flow_ids_matching(is_water_release)
withdrawal_rows = matrix_rows(water_withdrawal_flow_ids)
release_rows = matrix_rows(water_release_flow_ids)

# Then identify process columns that are active 
# in the solved product system.
active_columns = {
    column
    for _, column in base_lca.dicts.activity.items()
    if abs(float(base_lca.supply_array[column])) > active_supply_cutoff
}

# A candidate process must be active and must have at least one non-zero
# withdrawal row and one non-zero release row in the biosphere matrix.
candidate_columns = (
    nonzero_columns(withdrawal_rows)
    & nonzero_columns(release_rows)
    & active_columns
)

# Convert candidate matrix columns back to Brightway activity ids.
candidate_activity_ids = [
    base_lca.dicts.activity.reversed[column]
    for column in sorted(candidate_columns)
]


def grouped_amounts(exchanges):
    grouped = defaultdict(float)
    for exchange in exchanges:
        grouped[exchange.input.id] += float(exchange.get("amount"))
    return dict(grouped)


# Only now do we inspect exchange objects. The matrix gave us the candidate
# columns, but exchange objects hold the uncertainty fields needed later.
withdrawal_flow_id_set = set(water_withdrawal_flow_ids)
release_flow_id_set = set(water_release_flow_ids)
water_balance_groups = []

for activity_id in candidate_activity_ids:
    activity_node = bd.get_node(id=activity_id)
    exchanges = list(activity_node.biosphere())
    withdrawals = [exc for exc in exchanges if exc.input.id in withdrawal_flow_id_set]
    releases = [exc for exc in exchanges if exc.input.id in release_flow_id_set]

    total_withdrawal = sum(float(exc.get("amount")) for exc in withdrawals)
    total_release = sum(float(exc.get("amount")) for exc in releases)
    if total_withdrawal <= 0 or total_release < 0:
        continue

    water_balance_groups.append({
        "activity": activity_node,
        "activity_id": activity_id,
        "withdrawals": withdrawals,
        "releases": releases,
        "withdrawal_amounts": grouped_amounts(withdrawals),
        "release_amounts": grouped_amounts(releases),
        "total_withdrawal": total_withdrawal,
        "total_release": total_release,
        "return_fraction": total_release / total_withdrawal,
    })

if not water_balance_groups:
    raise ValueError("No active processes with both water withdrawals and releases were found.")

summary = pd.DataFrame([
    {
        "activity": group["activity"].get("name"),
        "location": group["activity"].get("location"),
        "withdrawal rows": len(group["withdrawal_amounts"]),
        "release rows": len(group["release_amounts"]),
        "total withdrawal": group["total_withdrawal"],
        "total release": group["total_release"],
        "return fraction": group["return_fraction"],
    }
    for group in water_balance_groups
])

print("Water withdrawal flow rows:", len(withdrawal_rows))
print("Water release flow rows:", len(release_rows))
print("Candidate active process columns:", len(candidate_activity_ids))
print("Processes with usable water balances:", len(water_balance_groups))
print("Matrix coordinates to override:", int(summary["withdrawal rows"].sum() + summary["release rows"].sum()))
display(summary.head(10))


## 4) First run: independent Brightway sampling

This is the default behavior. We track one example process to show the problem: the release-to-withdrawal ratio changes because the withdrawal and release matrix entries are sampled independently.


In [ ]:
def matrix_return_fraction(lca, group):
    # Read the actual values currently present in `lca.biosphere_matrix`.
    # In a Monte Carlo LCA these values change after each `next(lca)` call.
    column = lca.dicts.activity[group["activity_id"]]
    withdrawal_total = sum(
        float(lca.biosphere_matrix[lca.dicts.biosphere[flow_id], column])
        for flow_id in group["withdrawal_amounts"]
    )
    release_total = sum(
        float(lca.biosphere_matrix[lca.dicts.biosphere[flow_id], column])
        for flow_id in group["release_amounts"]
    )
    return release_total / withdrawal_total


# Track one process in detail so we can see the problem directly.
example_group = water_balance_groups[0]

# This is the baseline: Brightway samples every uncertain exchange independently.
independent_lca = bc.LCA(
    demand,
    data_objs=data_objs,
    remapping_dicts=remapping,
    use_distributions=True,
)
independent_lca.lci()
independent_lca.lcia()

independent_records = []
for iteration in range(iterations):
    # If withdrawal and release are sampled independently, this ratio drifts.
    independent_records.append({
        "iteration": iteration,
        "return fraction": matrix_return_fraction(independent_lca, example_group),
        "score": float(independent_lca.score),
    })
    if iteration < iterations - 1:
        next(independent_lca)

independent_results = pd.DataFrame(independent_records)

In [ ]:
plt.hist(independent_results["score"], bins=100)[-1]

## 5) Build system-wide correlated water samples

For each process, we sample the withdrawal side and derive the release side. The rule is:

1. Use one shuffled unit-interval sequence per process.
2. Sample each withdrawal exchange from its own uncertainty distribution using that sequence.
3. Sum withdrawal samples to get the sampled total withdrawal for that process.
4. Scale every release flow by `sampled_total_withdrawal / deterministic_total_withdrawal`.

This preserves the original return fraction and release split inside each process. Different processes get different shuffled sequences, so this only imposes correlation within each process.


In [ ]:
def uncertainty_type(exchange):
    # Brightway stores exchange uncertainty using stats_arrays fields.
    # Missing or zero uncertainty means deterministic amount.
    return int(exchange.get("uncertainty type") or exchange.get("uncertainty_type") or 0)


def sample_exchange_from_unit_interval(exchange, unit_interval):
    # We use inverse-CDF sampling (`ppf`) so several exchanges can share the
    # same unit-interval sequence and therefore move together.
    if uncertainty_type(exchange) == 0:
        return np.full(unit_interval.size, float(exchange.get("amount")))

    params = UncertaintyBase.from_dicts(exchange.as_dict())
    distribution = uncertainty_choices[int(params["uncertainty_type"][0])]
    samples = distribution.ppf(params, unit_interval.reshape(1, -1)).reshape(-1)
    if not np.all(np.isfinite(samples)):
        raise ValueError(f"Non-finite samples for exchange {exchange}")
    return samples


rng = np.random.default_rng(seed)
base_unit_interval = (np.arange(iterations) + 0.5) / iterations

# These two lists become the bw_processing array:
# - `indices` says which biosphere-matrix coordinate to override;
# - `data_rows` gives one sampled value per Monte Carlo iteration.
indices = []
data_rows = []
records = []
balance_errors = []

for group_number, group in enumerate(water_balance_groups):
    # One shuffled quantile sequence per process. All withdrawals in this
    # process use the same sequence, so they are correlated within the process.
    unit_interval = base_unit_interval.copy()
    rng.shuffle(unit_interval)

    withdrawal_samples_by_flow = defaultdict(lambda: np.zeros(iterations))
    total_withdrawal_samples = np.zeros(iterations)

    # Sample the withdrawal side from the original exchange uncertainties.
    for exchange in group["withdrawals"]:
        samples = sample_exchange_from_unit_interval(exchange, unit_interval)
        withdrawal_samples_by_flow[exchange.input.id] += samples
        total_withdrawal_samples += samples

    # This scale factor is the core idea. If sampled withdrawals are 20% above
    # their deterministic total, releases are also set 20% above their amounts.
    release_scale = total_withdrawal_samples / group["total_withdrawal"]

    # Validate the balance directly on the arrays. This is much cheaper than
    # checking every process inside every Monte Carlo LCA iteration later.
    sampled_release_total = group["total_release"] * release_scale
    sampled_return_fraction = sampled_release_total / total_withdrawal_samples
    balance_errors.append(
        np.max(np.abs(sampled_return_fraction - group["return_fraction"]))
    )

    # Store sampled withdrawals.
    for flow_id, samples in withdrawal_samples_by_flow.items():
        indices.append((flow_id, group["activity_id"]))
        data_rows.append(samples)

    # Store derived releases. This preserves the process-level return fraction.
    for flow_id, deterministic_amount in group["release_amounts"].items():
        indices.append((flow_id, group["activity_id"]))
        data_rows.append(deterministic_amount * release_scale)

    records.append({
        "group": group_number,
        "activity": group["activity"].get("name"),
        "location": group["activity"].get("location"),
        "return fraction": group["return_fraction"],
        "matrix rows": len(withdrawal_samples_by_flow) + len(group["release_amounts"]),
    })

# The datapackage is sequential: iteration 0 uses column 0 of `water_data`,
# iteration 1 uses column 1, and so on.
dp_water = bwp.create_datapackage(sequential=True)
water_data = np.vstack(data_rows)
water_indices = np.array(indices, dtype=bwp.INDICES_DTYPE)
dp_water.add_persistent_array(
    matrix="biosphere_matrix",
    indices_array=water_indices,
    data_array=water_data,
    name="system-wide-correlated-water-balances",
)
override_table = pd.DataFrame(records)
max_presampled_balance_error = float(np.max(balance_errors))

print(f"Seed: {seed}; iterations: {iterations}")
print("Processes controlled:", len(override_table))
print("Matrix coordinates controlled:", water_data.shape[0])
print("Max pre-sampled return-fraction error:", max_presampled_balance_error)
display(override_table.head(10))


## 6) Second run: controlled water balances, normal sampling elsewhere

We pass the normal Brightway datapackages plus our system-wide water-balance datapackage. `use_arrays=True` activates the pre-sampled values, while `use_distributions=True` keeps the usual uncertainty distributions active for all other uncertain matrix entries.


In [ ]:
# Add the custom datapackage after the normal Brightway datapackages.
# `use_arrays=True` activates our pre-sampled values.
# `use_distributions=True` keeps all other uncertain exchanges stochastic.
controlled_lca = bc.LCA(
    demand,
    data_objs=data_objs + [dp_water],
    remapping_dicts=remapping,
    use_distributions=True,
    use_arrays=True,
)
controlled_lca.lci()
controlled_lca.lcia()

controlled_records = []
for iteration in range(iterations):
    # Keep this loop light. The full balance validation was already done on the
    # pre-sampled arrays; here we only record the score and one example ratio.
    controlled_records.append({
        "iteration": iteration,
        "example return fraction": matrix_return_fraction(controlled_lca, example_group),
        "score": float(controlled_lca.score),
    })

    if iteration < iterations - 1:
        next(controlled_lca)

controlled_results = pd.DataFrame(controlled_records)

## 7) Compare the two Monte Carlo score distributions

The controlled run still has stochastic scores because the rest of the inventory can still be sampled. The difference is that every matched process-level water return fraction is now fixed. The plot shows the uncontrolled and controlled score distributions side by side with common bin edges.


In [ ]:
independent_scores = independent_results["score"].to_numpy()
controlled_scores = controlled_results["score"].to_numpy()

independent_bins = np.histogram_bin_edges(independent_scores, bins=30)
controlled_bins = np.histogram_bin_edges(controlled_scores, bins=30)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

axes[0].hist(
    independent_scores,
    bins=independent_bins,
    alpha=0.8,
    color="tab:orange",
)
axes[0].axvline(base_lca.score, color="black", linestyle="--")
axes[0].set_xlim(independent_scores.min(), independent_scores.max())
axes[0].set_title("Uncontrolled sampling")
axes[0].set_xlabel("Impact score")
axes[0].set_ylabel("Iterations")

axes[1].hist(
    controlled_scores,
    bins=controlled_bins,
    alpha=0.8,
    color="tab:blue",
)
axes[1].axvline(base_lca.score, color="black", linestyle="--", label="Deterministic score")
axes[1].set_xlim(controlled_scores.min(), controlled_scores.max())
axes[1].set_title("Controlled water balances")
axes[1].set_xlabel("Impact score")
axes[1].legend()

fig.suptitle("Monte Carlo score distributions")
plt.tight_layout()
plt.show()
